# Topography Effects

In [17]:
from simpeg.utils import mkvc, ndgrid, plot2Ddata

# Basic Python functionality
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FormatStrFormatter

mpl.rcParams.update({"font.size": 14})

from ipywidgets import (
    interact,
    interactive,
    IntSlider,
    widget,
    FloatText,
    FloatSlider,
    fixed,
)

In [18]:
survey_dir = './part_1_outputs/'
dobs_dir = './part_2_outputs/'  # Want in SimPEG convention
dpre_dir = './part_4_outputs/'

# Wasn't simulated for all frequencies
fmin, fmax = 10, 1000

# Wasn't simulated on the whole area
xmin, xmax = 3475., 1e8
ymin, ymax = 2475., 1e8

## Load Data

In [20]:
frequencies_mt = np.load(survey_dir + 'frequencies_mt.npy')
locations_mt = np.load(survey_dir + 'locations_mt.npy')
dobs_mt = np.load(dobs_dir + 'dtrue_mt_simpeg.npy').reshape((len(frequencies_mt), 8, len(locations_mt)))
dpre_mt = np.load(dpre_dir + 'dpred_mt.npy')

inds_loc = (locations_mt[:, 0] >= xmin) & (locations_mt[:, 0] <= xmax) & (locations_mt[:, 1] >= ymin) & (locations_mt[:, 1] <= ymax)
inds_freq = (frequencies_mt >= fmin) & (frequencies_mt <= fmax)
dobs_mt = dobs_mt[inds_freq, :, :]
dobs_mt = dobs_mt[:, :, inds_loc]
frequencies_mt = frequencies_mt[inds_freq]
locations_mt = locations_mt[inds_loc, :]

n_freq_mt, n_comp_mt, n_loc_mt = np.shape(dpre_mt)

In [21]:
frequencies_ztem = np.load(survey_dir + 'frequencies_ztem.npy')
locations_ztem = np.load(survey_dir + 'locations_ztem.npy')
dobs_ztem = np.load(dobs_dir + 'dtrue_ztem_simpeg.npy').reshape((len(frequencies_ztem), 4, len(locations_ztem)))
dpre_ztem = np.load(dpre_dir + 'dpred_ztem.npy')

inds_loc = (locations_ztem[:, 0] >= xmin) & (locations_ztem[:, 0] <= xmax) & (locations_ztem[:, 1] >= ymin) & (locations_ztem[:, 1] <= ymax)
inds_freq = (frequencies_ztem >= fmin) & (frequencies_ztem <= fmax)
dobs_ztem = dobs_ztem[inds_freq, :, :]
dobs_ztem = dobs_ztem[:, :, inds_loc]
frequencies_ztem = frequencies_ztem[inds_freq]
locations_ztem = locations_ztem[inds_loc, :]

n_freq_ztem, n_comp_ztem, n_loc_ztem = np.shape(dpre_ztem)

In [22]:
frequencies_appcon = np.load(survey_dir + 'frequencies_appcon.npy')
locations_appcon = np.load(survey_dir + 'locations_appcon.npy')
dobs_appcon = np.load(dobs_dir + 'dtrue_appcon.npy').reshape((len(frequencies_appcon), len(locations_appcon)))
dpre_appcon = np.load(dpre_dir + 'dpred_appcon.npy')

inds_loc = (locations_appcon[:, 0] >= xmin) & (locations_appcon[:, 0] <= xmax) & (locations_appcon[:, 1] >= ymin) & (locations_appcon[:, 1] <= ymax)
inds_freq = (frequencies_appcon >= fmin) & (frequencies_appcon <= fmax)
dobs_appcon = dobs_appcon[inds_freq, :]
dobs_appcon = dobs_appcon[:, inds_loc]
frequencies_appcon = frequencies_appcon[inds_freq]
locations_appcon = locations_appcon[inds_loc, :]

n_freq_appcon, n_loc_appcon = np.shape(dpre_appcon)

## Plot MT

In [61]:
mpl.rcParams.update({'font.size': 13})

d_list_mt = [dobs_mt, dpre_mt, dobs_mt - dpre_mt]
mt_type_list = ['Re[Zxx]', 'Im[Zxx]', 'Re[Zxy]', 'Im[Zxy]', 'Re[Zyx]', 'Im[Zyx]', 'Re[Zyy]', 'Im[Zyy]']

def plot_mt(f_ind, rx_ind):
    
    f_ind = f_ind - 1
    rx_ind = rx_ind - 1

    fig = plt.figure(figsize=(12, 4))

    ax1 = 3 * [None]
    ax2 = 3 *[None]
    norm = 3 * [None]
    cs = 3 * [None]
    cplot = 3 * [None]
    cbar = 3 * [None]
    cmap = 3 * [None]

    COUNT = 0

    for ii in range(3):  # true, obs, noise

        ax1[COUNT] = fig.add_axes([0.05+0.3*ii, 0.25, 0.25, 0.7])
        ax2[COUNT] = fig.add_axes([0.07+0.3*ii, 0.05, 0.21, 0.05])

        if ii == 0:
            data_temp = d_list_mt[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtrue' 
        elif ii == 1:
            data_temp = d_list_mt[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtopo'
        if ii == 2:
            data_temp = d_list_mt[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'diff'

        norm[COUNT]= mpl.colors.Normalize(vmin=vmin, vmax=vmax)

        ax1[COUNT].scatter(locations_mt[:, 0], locations_mt[:, 1], 108, c=data_temp, cmap=mpl.cm.Spectral_r, norm=norm[COUNT])

        ax1[COUNT].set_title(title + ": {}, {} Hz".format(mt_type_list[rx_ind], frequencies_mt[f_ind]))
        ax1[COUNT].set_xlabel('Easting (m)')
        if ii == 0:
            ax1[COUNT].set_ylabel('Northing (m)')
        else:
            ax1[COUNT].set_yticks([])

        cbar[COUNT]= mpl.colorbar.ColorbarBase(ax2[COUNT], norm=norm[COUNT], orientation="horizontal", cmap=cmap[COUNT])
        cbar[COUNT].set_label('')
        ax2[COUNT].set_xticks([vmin, 0.5*(vmin+vmax), vmax])
        
        COUNT = COUNT + 1

    Q = np.sqrt(np.pi*frequencies_mt[f_ind]*4*np.pi*1e-7 / 0.02)
    print(Q)

def DataWidgetMT():

    i = interact(
        plot_mt,
        f_ind=IntSlider(
            min=1,
            max=n_freq_mt,
            value=1,
            step=1,
            continuous_update=False,
            description="FREQID",
        ),
        rx_ind=IntSlider(
            min=1,
            max=n_comp_mt,
            value=1,
            step=1,
            continuous_update=False,
            description="RXID",
        ),
    )
    
    return i

In [62]:
DataWidgetMT()

interactive(children=(IntSlider(value=1, continuous_update=False, description='FREQID', max=21, min=1), IntSli…

<function __main__.plot_mt(f_ind, rx_ind)>

## Plot ZTEM

In [51]:
mpl.rcParams.update({'font.size': 13})

d_list_ztem = [dobs_ztem, dpre_ztem, dobs_ztem - dpre_ztem]
ztem_type_list = ['Re[Tzx]', 'Im[Tzx]', 'Re[Tzy]', 'Im[Tzy]']

def plot_ztem(f_ind, rx_ind):
    
    f_ind = f_ind - 1
    rx_ind = rx_ind - 1

    fig = plt.figure(figsize=(12, 4))

    ax1 = 3 * [None]
    ax2 = 3 *[None]
    norm = 3 * [None]
    cs = 3 * [None]
    cplot = 3 * [None]
    cbar = 3 * [None]
    cmap = 3 * [None]

    COUNT = 0

    for ii in range(3):  # true, obs, noise

        ax1[COUNT] = fig.add_axes([0.05+0.3*ii, 0.25, 0.25, 0.7])
        ax2[COUNT] = fig.add_axes([0.07+0.3*ii, 0.05, 0.21, 0.05])

        if ii == 0:
            data_temp = d_list_ztem[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtrue' 
        elif ii == 1:
            data_temp = d_list_ztem[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtopo'
        if ii == 2:
            data_temp = d_list_ztem[ii][f_ind, rx_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'diff'

        norm[COUNT]= mpl.colors.Normalize(vmin=vmin, vmax=vmax)

        cplot[COUNT], ax1[COUNT] = plot2Ddata(
            locations_ztem[:, 0:2],
            data_temp,
            nx=200,
            ny=200,
            ax=ax1[COUNT],
            ncontour=200,
            contourOpts={"cmap": cmap[COUNT], "norm": norm[COUNT]},
        )

        ax1[COUNT].set_title(title + f": {ztem_type_list[rx_ind]}, {frequencies_ztem[f_ind]:.1f} Hz")
        ax1[COUNT].set_xlabel('Easting (m)')
        if ii == 0:
            ax1[COUNT].set_ylabel('Northing (m)')
        else:
            ax1[COUNT].set_yticks([])

        cbar[COUNT]= mpl.colorbar.ColorbarBase(ax2[COUNT], norm=norm[COUNT], orientation="horizontal", cmap=cmap[COUNT])
        cbar[COUNT].set_label('')
        ax2[COUNT].set_xticks([vmin, 0.5*(vmin+vmax), vmax])
        
        COUNT = COUNT + 1

def DataWidgetZTEM():

    i = interact(
        plot_ztem,
        f_ind=IntSlider(
            min=1,
            max=n_freq_ztem,
            value=1,
            step=1,
            continuous_update=False,
            description="FREQID",
        ),
        rx_ind=IntSlider(
            min=1,
            max=n_comp_ztem,
            value=1,
            step=1,
            continuous_update=False,
            description="RXID",
        ),
    )
    
    return i

In [52]:
DataWidgetZTEM()

interactive(children=(IntSlider(value=1, continuous_update=False, description='FREQID', max=6, min=1), IntSlid…

<function __main__.plot_ztem(f_ind, rx_ind)>

## Plot Apparent Conductivities

In [53]:
mpl.rcParams.update({'font.size': 13})

d_list_appcon = [dobs_appcon, dpre_appcon, dobs_appcon - dpre_appcon]

def plot_appcon(f_ind):
    
    f_ind = f_ind - 1

    fig = plt.figure(figsize=(12, 4))

    ax1 = 3 * [None]
    ax2 = 3 *[None]
    norm = 3 * [None]
    cs = 3 * [None]
    cplot = 3 * [None]
    cbar = 3 * [None]
    cmap = 3 * [None]

    COUNT = 0

    for ii in range(3):  # true, obs, noise

        ax1[COUNT] = fig.add_axes([0.05+0.3*ii, 0.25, 0.25, 0.7])
        ax2[COUNT] = fig.add_axes([0.07+0.3*ii, 0.05, 0.21, 0.05])

        if ii == 0:
            data_temp = d_list_appcon[ii][f_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtrue' 
        elif ii == 1:
            data_temp = d_list_appcon[ii][f_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'dtopo'
        if ii == 2:
            data_temp = d_list_appcon[ii][f_ind, :]
            cmap[COUNT] = mpl.cm.Spectral_r
            vmin, vmax = np.min(data_temp), np.max(data_temp)
            title = 'diff'

        norm[COUNT]= mpl.colors.Normalize(vmin=vmin, vmax=vmax)

        cplot[COUNT], ax1[COUNT] = plot2Ddata(
            locations_appcon[:, 0:2],
            data_temp,
            nx=200,
            ny=200,
            ax=ax1[COUNT],
            ncontour=200,
            contourOpts={"cmap": cmap[COUNT], "norm": norm[COUNT]},
        )

        ax1[COUNT].set_title(title + f": {frequencies_appcon[f_ind]:.1f} Hz")
        ax1[COUNT].set_xlabel('Easting (m)')
        if ii == 0:
            ax1[COUNT].set_ylabel('Northing (m)')
        else:
            ax1[COUNT].set_yticks([])

        cbar[COUNT]= mpl.colorbar.ColorbarBase(ax2[COUNT], norm=norm[COUNT], orientation="horizontal", cmap=cmap[COUNT])
        cbar[COUNT].set_label('')
        ax2[COUNT].set_xticks([vmin, 0.5*(vmin+vmax), vmax])
        
        COUNT = COUNT + 1

def DataWidgetAppCon():

    i = interact(
        plot_appcon,
        f_ind=IntSlider(
            min=1,
            max=n_freq_appcon,
            value=1,
            step=1,
            continuous_update=False,
            description="FREQID",
        ),
    )
    
    return i

In [54]:
DataWidgetAppCon()

interactive(children=(IntSlider(value=1, continuous_update=False, description='FREQID', max=16, min=1), Output…

<function __main__.plot_appcon(f_ind)>